In [1]:
%pip install jiwer
%pip install evaluate
%pip install -U accelerate
%pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128
%pip install -U transformers
%pip install librosa 
%pip install pandas
%pip install tensorboard
%pip install numpy==1.24.2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/nightly/cu128
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



In [2]:
import torch
import torchaudio
import gc
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2CTCTokenizer, Wav2Vec2ForCTC, Wav2Vec2Processor
from huggingface_hub import login

login(token="REMOVED_SECRET")

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

main_seed=42
model_name = "facebook/mms-1b"
model_language = "mansi"

path = "data/"

dataset = load_dataset('csv', data_files=path+'dataset.csv')

In [4]:
import json

transcripts = dataset['train']['sentence']

translation_table = dict.fromkeys(map(ord, '":;«¬»–—…\'̄().?!-,0123456789'), None)
transcripts = [line.translate(translation_table).lower().strip() for line in transcripts]

all_text = " ".join(transcripts)
vocab = list(set(all_text))

vocab_dict = {char: i for i, char in enumerate(vocab)}
vocab_dict['|'] = vocab_dict[' ']
del vocab_dict[' ']
vocab_dict["[PAD]"] = len(vocab_dict)
vocab_dict["[UNK]"] = len(vocab_dict)

with open('vocab.json', 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

tokenizer = Wav2Vec2CTCTokenizer("./vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")

In [5]:
tokenizer.get_vocab()

{'е': 0,
 'я': 1,
 'ō': 2,
 'ъ': 3,
 'ц': 4,
 'к': 5,
 'и': 6,
 'ю': 7,
 'в': 8,
 'л': 9,
 'з': 10,
 'м': 12,
 'о': 13,
 'ы': 14,
 'ā': 15,
 'р': 16,
 'ч': 17,
 'ж': 18,
 'н': 19,
 'э': 20,
 'б': 21,
 'щ': 22,
 'т': 23,
 'й': 24,
 'ё': 25,
 'г': 26,
 'д': 27,
 'ь': 28,
 'х': 29,
 'ӯ': 30,
 'ē': 31,
 'а': 32,
 'с': 33,
 'у': 34,
 'ш': 35,
 'ӣ': 36,
 'ф': 37,
 'п': 38,
 'ӈ': 39,
 '|': 11,
 '[PAD]': 40,
 '[UNK]': 41,
 '<s>': 42,
 '</s>': 43}

In [6]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def feature_extract(batch):
    speech_array, sampling_rate = torchaudio.load(path+batch['file'])
    speech = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)(speech_array).squeeze().numpy()
    batch["input_values"] = feature_extractor(speech, sampling_rate=16000).input_values[0]
    return batch


def tokenize(batch):
    text = batch['sentence'].translate(translation_table).lower().strip()
    with processor.as_target_processor():
        batch["labels"] = processor(text).input_ids

    return batch

dataset = dataset.map(feature_extract)

dataset = dataset.map(tokenize)

# dataset = dataset.map(feature_extract, num_proc=4)

# dataset = dataset.map(tokenize, num_proc=4)

dataset.set_format(type='torch', columns=['input_values', 'labels'])
dataset = dataset['train'].train_test_split(test_size=0.05, seed=main_seed)

dataset["train"].features

Map:   0%|          | 0/8997 [00:00<?, ? examples/s]

Map:   0%|          | 0/8997 [00:00<?, ? examples/s]

C:\Python396\lib\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


{'id': Value(dtype='int64', id=None),
 'sentence': Value(dtype='string', id=None),
 'file': Value(dtype='string', id=None),
 'input_values': Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None),
 'labels': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None)}

In [7]:
# import IPython.display as ipd
# import numpy as np
# import random

# rand_int = random.randint(0, len(dataset["train"]))

# print(tokenizer.decode(dataset["train"]["labels"][rand_int]))
# ipd.Audio(data=np.asarray(dataset["train"]["input_values"][rand_int]), autoplay=True, rate=16000)


In [8]:
import torch

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
        max_length (:obj:`int`, `optional`):
            Maximum length of the ``input_values`` of the returned list and optionally padding length (see above).
        max_length_labels (:obj:`int`, `optional`):
            Maximum length of the ``labels`` returned list and optionally padding length (see above).
        pad_to_multiple_of (:obj:`int`, `optional`):
            If set will pad the sequence to a multiple of the provided value.
            This is especially useful to enable the use of Tensor Cores on NVIDIA hardware with compute capability >=
            7.5 (Volta).
    """

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    max_length_labels: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    pad_to_multiple_of_labels: Optional[int] = None

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                max_length=self.max_length_labels,
                pad_to_multiple_of=self.pad_to_multiple_of_labels,
                return_tensors="pt",
            )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [9]:
import evaluate
import numpy as np

metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")

def compute_metrics(pred):
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    random_idx = np.random.randint(0, len(pred_ids))

    print("Label IDs:", label_ids[random_idx])
    print("Predicted IDs:", pred_ids[random_idx])
    print("Label text:", label_str[random_idx])
    print("Predicted text:", pred_str[random_idx])

    wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)

    print("norm_wer: " + str(wer))
    print("norm_cer: " + str(cer))

    return {"norm_wer": wer, "norm_cer": cer}

In [10]:
model = Wav2Vec2ForCTC.from_pretrained(
    model_name ,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size = processor.tokenizer.vocab_size
)

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
from transformers import TrainingArguments, Trainer

model.freeze_feature_extractor()

training_args = TrainingArguments(
    output_dir="./mms-300m-mansi",
    overwrite_output_dir=True,
    save_total_limit=2,

    # learning_rate=1e-4,
    # weight_decay=0.005,
    # optim='adafactor',
    group_by_length=True,
    gradient_checkpointing=True,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=15,
    save_strategy="epoch",
    eval_strategy="epoch",

    seed=main_seed,
    data_seed=main_seed,
    fp16=torch.cuda.is_available(),
    fp16_full_eval=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="norm_wer",
    greater_is_better=False,
    report_to=["tensorboard"],
    push_to_hub=True,
    hub_private_repo=True,
    hub_always_push=True,
)

trainer = Trainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

C:\Python396\lib\site-packages\transformers\models\wav2vec2\modeling_wav2vec2.py:2175: FutureWarning: The method `freeze_feature_extractor` is deprecated and will be removed in Transformers v5. Please use the equivalent `freeze_feature_encoder` method instead.
  warnings.warn(


In [12]:
trainer.train()
trainer.push_to_hub()

C:\Python396\lib\site-packages\transformers\utils\generic.py:271: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  arr = np.array(obj)


Epoch,Training Loss,Validation Loss,Norm Wer,Norm Cer
1,No log,0.806783,70.394737,20.617234
2,2.155700,0.594471,58.176692,16.721443
3,2.155700,0.529676,54.511278,15.751503
4,0.768700,0.484652,51.926692,15.062124
5,0.768700,0.458974,49.154135,14.460922
6,0.529000,0.480271,50.000000,14.685371
7,0.529000,0.459004,49.201128,14.404810
8,0.455500,0.436785,47.979323,13.963928
9,0.455500,0.408637,47.979323,13.859719
10,0.412300,0.411178,46.945489,13.931864


Label IDs: [12  6 19  1 11 12  6 19 20 19 11  9 32 29 14 33 11 32 23 20 19 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40]
Predicted IDs: [40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 12 12 40 40 40 40 40 40 19 40 11 40 40 40  1 40 40 40 40 40 11 11 12
 12  6 40 40 40 40 40 40 40 19 19 20 40 40 40 40 40 19 11 11 40 40 40  9
 32 40 40 40 40 40 40 40 40 40 40 40 40 29 29 40 40 40 33 40 11 40 40 40
 40 32 32 40 40 40 40 40 40 40 23 23 20 40 40 40 40 40 40 40 19 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0

C:\Python396\lib\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
C:\Python396\lib\site-packages\transformers\utils\generic.py:271: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  arr = np.array(obj)


Label IDs: [29 13 33 32 23 11 23  6  1 11 13  9 14 33 11 11 32  5  8 19 32  5 23 11
 13  8 14  9 23 32 29 23 30 12 11 23 30  1 40 40 40 40 40 40 40]
Predicted IDs: [40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 23 40  6 40 40 40 40 40
 40 23 40 40 40 40 29 40 34 40 40 40 40 40 40 12 11 11 40 40 12 40  6 40
 40 40 40 40 40 19 20 40 40 40 40 40 40 26 14 40 40 40 40 11 11 40 40 40
 40 40 40 40 33 32 40 40 40 40 40 40 40 40  5  8 40 40 40 40 40 40 40 40
 38 38 32 40 40 40 40 40 40 40 40 40 40 26 26 40 40 40 40  9 14 40 40 40
 40 40 11 11 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 33 32 40 40 40 40 40 40 40 40  5  8 11 40 40 29 13 40 40 40 40
 40 40 40 40 33 14 40 40 40 40 40 23 40 11 40 40 40 12  6 40 40 40 40 40
 19 20 40 40 40 40 40 40 40 40 26 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40

...

C:\Python396\lib\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
C:\Python396\lib\site-packages\transformers\utils\generic.py:271: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  arr = np.array(obj)


Label IDs: [29 13 23 32  9 20 11 33 32 23 11 30  9 34 12 23 32  9 11 13  9 34 12 11
 29 34 12 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40]
Predicted IDs: [40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 29 13 40 40 40 40 40
 40 40 40 40 23 32 40 40 40 40 40 40  9 40 20 40 40 40 40 40 40 11 11 40
 40 40 33 32 40 40 40 40 40 40 40 40 40 40 23 40 11 11 40 40 40 40 40 40
 40 40 40 40 40 40 30 40 40 40 40 40 40 40 40  9 40 34 40 40 40 40 40 40
 12 40 40 40 40 23 32 40 40 40 40 40 40  9 40 11 11 40 40 40 13 40 40 40
 40 40 40 40 40 40  9 34 40 40 40 40 40 40 12 40 11 11 40 40 40 29 40 34
 40 40 40 40 40 40 40 40 40 12 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40

C:\Python396\lib\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
C:\Python396\lib\site-packages\transformers\utils\generic.py:271: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  arr = np.array(obj)


Label IDs: [38 14 26 11  5 32 33 14 39 11  5 13  9 23 32 11  5 31 19 28 33  1 12 23
 32  8  0 12 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40]
Predicted IDs: [40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 38 40 14 40 40 40 40 40 40 40 40 26 40 40 40 11 40 40 40 40  5 40
 32 40 40 40 40 40 40 40 40 40 40 40 33 14 40 40 40 40 40 40 39 40 11 11
 40 40 40 40  5 40 13 40 40 40 40 40 40 40 40 23 40 40 40 40  9 32 40 40
 40 40 40 40 40 11 11 11 40 40 40 40 40 40 40 40 40 40 40 40 40  5 40 40
 31 40 40 40 40 40 40 19 40 40 40 28 40 40 40 40 33 40  1 40 40 40 40 40
 40 12 40 40 40 40 40 40 40 23 32 40 40 40 40 40 40 40 40  8  0 40 40 40
 40 40 40 12 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40 40
 40 40 40 40 40 40 40 40 40 40 40

HfHubHTTPError: (Request ID: Root=1-6817c3b7-0e355e0802ad7d846bc045f7;ba3a9732-6cd5-46ac-b1ac-9048b06bd761)

403 Forbidden: Private repository storage limit reached, please upgrade your plan to increase your private storage limit.
Cannot access content at: https://huggingface.co/AMRPH/mms-300m-mansi.git/info/lfs/objects/batch.
Make sure your token has the correct permissions.

In [13]:
trainer.push_to_hub()

events.out.tfevents.1746382852.viraly.1388.0:   0%|          | 0.00/6.37k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.71k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.85G [00:00<?, ?B/s]

Upload 5 LFS files:   0%|          | 0/5 [00:00<?, ?it/s]

events.out.tfevents.1746381462.viraly.15848.0:   0%|          | 0.00/6.37k [00:00<?, ?B/s]

events.out.tfevents.1746383503.viraly.20100.0:   0%|          | 0.00/14.0k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/AMRPH/mms-300m-mansi/commit/8ea32ed4d669ff837c2fd56c4f59d4558eb18ae6', commit_message='End of training', commit_description='', oid='8ea32ed4d669ff837c2fd56c4f59d4558eb18ae6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/AMRPH/mms-300m-mansi', endpoint='https://huggingface.co', repo_type='model', repo_id='AMRPH/mms-300m-mansi'), pr_revision=None, pr_num=None)